# LSTM Forecasting Baseline

## Energy Time-Series Transfer Learning Project

This notebook establishes the forecasting baseline before introducing
Transformer models and transfer learning.

### Research setup

- **Source task:** PV generation forecasting
- **Target task:** Household grid-import forecasting
- **Forecast horizon:** next hour
- **Input window:** previous 24 hours
- **Model:** LSTM
- **Evaluation:** chronological train/validation/test split

The modelling data were prepared in Notebook 01 from cumulative energy
measurements. The variables used here are hourly energy in kWh.


## Why an LSTM baseline?

The planned research direction uses Transformer-based time-series forecasting
and transfer learning. An LSTM provides a strong and interpretable recurrent
baseline.

The experiment will compare:

1. Persistence baseline
2. LSTM forecasting

The same pipeline is used for PV and grid-import forecasting so that the
results can later be extended to the transfer-learning experiment.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)


In [ ]:
# Google Colab path created by Notebook 01

DATA_PATH = "/content/residential4_model_data.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])

df = df.sort_values("timestamp").reset_index(drop=True)

print("Shape:", df.shape)
print("Start:", df["timestamp"].min())
print("End:", df["timestamp"].max())

df.head()


In [ ]:
# Keep only the two primary forecasting series.

data = df[
    ["timestamp", "pv_hourly", "grid_import_hourly"]
].copy()

print(data.isna().sum())
print("\nRows:", len(data))


## 1. Chronological train/validation/test split

Time-series data must not be randomly shuffled before splitting because that
would allow future information to influence the training process.

We use:

- **70% training**
- **15% validation**
- **15% test**

All preprocessing parameters are fitted using the training period only.


In [ ]:
n = len(data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = data.iloc[:train_end].copy()
val = data.iloc[train_end:val_end].copy()
test = data.iloc[val_end:].copy()

print("Train:", train["timestamp"].min(), "→", train["timestamp"].max())
print("Validation:", val["timestamp"].min(), "→", val["timestamp"].max())
print("Test:", test["timestamp"].min(), "→", test["timestamp"].max())

print("\nSizes:")
print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))


In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(
    train["timestamp"],
    train["pv_hourly"],
    linewidth=0.7,
    label="Training"
)

plt.plot(
    val["timestamp"],
    val["pv_hourly"],
    linewidth=0.7,
    label="Validation"
)

plt.plot(
    test["timestamp"],
    test["pv_hourly"],
    linewidth=0.7,
    label="Test"
)

plt.xlabel("Time")
plt.ylabel("PV generation (kWh)")
plt.title("Chronological Dataset Split – PV")
plt.legend()

plt.tight_layout()
plt.show()


## 2. Evaluation metrics

We use **MAE** and **RMSE** as the main metrics.

MAPE is avoided as a primary metric because PV generation contains many
zero-value nighttime observations, making percentage errors unstable or
undefined.

We also report normalized RMSE relative to the mean target value for easier
comparison between experiments.


In [ ]:
def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    mean_target = np.mean(y_true)
    nrmse = rmse / mean_target if mean_target != 0 else np.nan

    return {
        "MAE": mae,
        "RMSE": rmse,
        "nRMSE": nrmse
    }


def print_metrics(name, y_true, y_pred):
    metrics = regression_metrics(y_true, y_pred)

    print(name)
    print("-" * len(name))
    print(f"MAE:   {metrics['MAE']:.4f} kWh")
    print(f"RMSE:  {metrics['RMSE']:.4f} kWh")
    print(f"nRMSE: {metrics['nRMSE']:.2%}")


## 3. Persistence baseline

The persistence model predicts that the next hour will equal the current
hour, using the final observation before the test period for the first test forecast.

For a one-step-ahead forecast:

**ŷ(t+1) = y(t)**

This is an important benchmark because an ML model should demonstrate a clear
improvement over this simple forecasting strategy.


In [ ]:
def persistence_forecast(train_val_series, test_series):
    """
    One-step persistence forecast.

    The first test prediction uses the final observation immediately
    before the test period.
    """
    train_val_series = np.asarray(train_val_series)
    test_series = np.asarray(test_series)

    previous_values = np.concatenate([
        [train_val_series[-1]],
        test_series[:-1]
    ])

    return test_series, previous_values


# Evaluate persistence on the test period.
for target in ["pv_hourly", "grid_import_hourly"]:
    y_true, y_pred = persistence_forecast(
        val[target].values,
        test[target].values
    )

    print_metrics(
        f"Persistence – {target}",
        y_true,
        y_pred
    )
    print()


## 4. Prepare sequences for the LSTM

The LSTM receives the previous **24 hourly observations** and predicts the
next hour.

Example:

`hours t-23 ... t → predict t+1`

The scaler is fitted only on the training data to avoid data leakage.


In [ ]:
LOOKBACK = 24

def make_sequences(values, lookback=24):
    values = np.asarray(values)

    X = []
    y = []

    for i in range(lookback, len(values)):
        X.append(values[i-lookback:i])
        y.append(values[i])

    X = np.array(X)
    y = np.array(y)

    return X, y


## 5. LSTM experiment function

The following function:

1. Fits a scaler on the training period.
2. Creates 24-hour sequences.
3. Trains the LSTM.
4. Uses early stopping based on validation loss.
5. Converts predictions back to the original kWh scale.
6. Evaluates the test set.

The validation sequence includes the final 24 hours of the training period so
that the first validation prediction has the correct historical context.
Likewise, the test sequence receives the end of the validation period as
context. No future test values are used as input to earlier predictions.


In [ ]:
def build_lstm_model(lookback=24):
    model = Sequential([
        Input(shape=(lookback, 1)),
        LSTM(
            64,
            return_sequences=False
        ),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(1)
    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model


In [ ]:
def run_lstm_experiment(dataframe, target, lookback=24, epochs=30):
    values = dataframe[target].values.astype(float)

    n = len(values)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    # Fit scaler using training data only.
    scaler = StandardScaler()
    scaler.fit(values[:train_end].reshape(-1, 1))

    scaled = scaler.transform(values.reshape(-1, 1)).flatten()

    # Training sequences
    train_values = scaled[:train_end]
    X_train, y_train = make_sequences(train_values, lookback)

    # Validation sequences: include training history before validation.
    val_values = scaled[train_end-lookback:val_end]
    X_val, y_val = make_sequences(val_values, lookback)

    # Test sequences: include validation history before test.
    test_values = scaled[val_end-lookback:]
    X_test, y_test = make_sequences(test_values, lookback)

    # Reshape for LSTM: samples, time steps, features
    X_train = X_train.reshape(-1, lookback, 1)
    X_val = X_val.reshape(-1, lookback, 1)
    X_test = X_test.reshape(-1, lookback, 1)

    model = build_lstm_model(lookback)

    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=64,
        callbacks=[early_stopping],
        verbose=1
    )

    # Predictions in scaled space
    y_pred_scaled = model.predict(X_test, verbose=0).flatten()

    # Return to original kWh scale
    y_pred = scaler.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).flatten()

    y_true = scaler.inverse_transform(
        y_test.reshape(-1, 1)
    ).flatten()

    return model, history, y_true, y_pred


# 6. PV forecasting

We first train the LSTM on the **PV generation** series.

This will become the source forecasting task for the later transfer-learning
experiment.


In [ ]:
tf.keras.backend.clear_session()

pv_model, pv_history, pv_true, pv_pred = run_lstm_experiment(
    data,
    target="pv_hourly",
    lookback=LOOKBACK,
    epochs=30
)

print_metrics(
    "LSTM – PV test set",
    pv_true,
    pv_pred
)


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(pv_history.history["loss"], label="Training loss")
plt.plot(pv_history.history["val_loss"], label="Validation loss")

plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("LSTM Training History – PV")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Compare LSTM and persistence on a representative test window.

pv_persistence_true, pv_persistence_pred = persistence_forecast(
    val["pv_hourly"].values,
    test["pv_hourly"].values
)

test_timestamps = test["timestamp"].reset_index(drop=True)

plot_n = min(168, len(pv_true))

plt.figure(figsize=(15, 5))

plt.plot(
    test_timestamps.iloc[:plot_n],
    pv_true[:plot_n],
    label="Actual",
    linewidth=1.2
)

plt.plot(
    test_timestamps.iloc[:plot_n],
    pv_pred[:plot_n],
    label="LSTM",
    linewidth=1
)

plt.plot(
    test_timestamps.iloc[:plot_n],
    pv_persistence_pred[:plot_n],
    label="Persistence",
    linewidth=1
)

plt.xlabel("Time")
plt.ylabel("PV generation (kWh)")
plt.title("PV Forecasting – One-Week Test Window")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
print_metrics(
    "Persistence – PV test set",
    pv_persistence_true,
    pv_persistence_pred
)

print()

print_metrics(
    "LSTM – PV test set",
    pv_true,
    pv_pred
)


## 7. Grid-import forecasting

The same architecture is now trained independently on household grid import.

This is the **target task** that will later be used in the transfer-learning
experiment.


In [ ]:
tf.keras.backend.clear_session()

grid_model, grid_history, grid_true, grid_pred = run_lstm_experiment(
    data,
    target="grid_import_hourly",
    lookback=LOOKBACK,
    epochs=30
)

print_metrics(
    "LSTM – Grid-import test set",
    grid_true,
    grid_pred
)


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(grid_history.history["loss"], label="Training loss")
plt.plot(grid_history.history["val_loss"], label="Validation loss")

plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("LSTM Training History – Grid Import")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
grid_persistence_true, grid_persistence_pred = persistence_forecast(
    val["grid_import_hourly"].values,
    test["grid_import_hourly"].values
)

grid_test_timestamps = test["timestamp"].reset_index(drop=True)

plot_n = min(168, len(grid_true))

plt.figure(figsize=(15, 5))

plt.plot(
    grid_test_timestamps.iloc[:plot_n],
    grid_true[:plot_n],
    label="Actual",
    linewidth=1.2
)

plt.plot(
    grid_test_timestamps.iloc[:plot_n],
    grid_pred[:plot_n],
    label="LSTM",
    linewidth=1
)

plt.plot(
    grid_test_timestamps.iloc[:plot_n],
    grid_persistence_pred[:plot_n],
    label="Persistence",
    linewidth=1
)

plt.xlabel("Time")
plt.ylabel("Grid-import energy (kWh)")
plt.title("Grid-Import Forecasting – One-Week Test Window")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
print_metrics(
    "Persistence – Grid-import test set",
    grid_persistence_true,
    grid_persistence_pred
)

print()

print_metrics(
    "LSTM – Grid-import test set",
    grid_true,
    grid_pred
)


## 8. Results summary

The table below provides the first quantitative benchmark for the project.

The important question is not simply whether the LSTM has a low error, but
whether it improves on the persistence baseline. This establishes the
reference point for the Transformer and transfer-learning experiments.


In [ ]:
results = []

for task, y_true, persistence_pred, lstm_pred in [
    (
        "PV generation",
        pv_true,
        pv_persistence_pred,
        pv_pred
    ),
    (
        "Grid import",
        grid_true,
        grid_persistence_pred,
        grid_pred
    )
]:
    persistence_metrics = regression_metrics(
        y_true,
        persistence_pred
    )

    lstm_metrics = regression_metrics(
        y_true,
        lstm_pred
    )

    results.append({
        "Task": task,
        "Persistence MAE": persistence_metrics["MAE"],
        "LSTM MAE": lstm_metrics["MAE"],
        "Persistence RMSE": persistence_metrics["RMSE"],
        "LSTM RMSE": lstm_metrics["RMSE"],
        "Persistence nRMSE": persistence_metrics["nRMSE"],
        "LSTM nRMSE": lstm_metrics["nRMSE"]
    })

results_df = pd.DataFrame(results)

results_df


In [ ]:
# Percentage improvement of LSTM over persistence.

results_df["MAE improvement (%)"] = (
    100
    * (results_df["Persistence MAE"] - results_df["LSTM MAE"])
    / results_df["Persistence MAE"]
)

results_df["RMSE improvement (%)"] = (
    100
    * (results_df["Persistence RMSE"] - results_df["LSTM RMSE"])
    / results_df["Persistence RMSE"]
)

results_df


# Conclusions

This notebook establishes the first machine-learning benchmark for the
project.

### Completed

- Chronological train/validation/test split
- Leakage-safe feature scaling
- 24-hour input window
- Persistence baseline
- LSTM model in TensorFlow/Keras
- PV generation forecasting
- Household grid-import forecasting
- MAE, RMSE and normalized RMSE evaluation
- Visual comparison of actual, persistence and LSTM forecasts

### Research progression

**Notebook 01:** Data preparation and EDA  
↓  
**Notebook 02:** LSTM forecasting baseline  
↓  
**Notebook 03:** Transformer-based forecasting  
↓  
**Notebook 04:** Transfer learning: PV → grid import

The PV LSTM model produced here will provide the starting model for the
transfer-learning experiment. The key idea in Notebook 04 will be to test
whether knowledge learned from the data-rich PV forecasting task can improve
grid-import forecasting when the target task has limited training data.
